# New Responses Examples 🍿

Let's do a quick snack-sized walkthrough of chatsnack's Responses runtime.

This notebook is intentionally beginner-friendly and follows the flavor of the existing chatsnack notebooks.


## Setup

### Got snack?
Install chatsnack and make sure your OpenAI SDK supports the Responses API (`openai>=1.66.0`).


In [ ]:
!pip install -q chatsnack "openai>=1.66.0" python-dotenv


### Got keys?
Put your API key in a local `.env` file:

```
OPENAI_API_KEY=sk-...
```


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from chatsnack import Chat
from chatsnack.chat.mixin_params import ChatParams


## Classic runtime (chat completions)
By default, chatsnack uses the chat completions runtime.


In [ ]:
classic = Chat("You are a concise tutor.", model="gpt-5.4-mini")
print("Runtime adapter:", type(classic.runtime).__name__)
classic.ask("Explain prompt engineering in one sentence.")


## New runtime (responses)
Now switch to the new Responses runtime with one argument.


In [ ]:
fresh = Chat(
    "You are a concise tutor.",
    runtime_selector="responses",
    model="gpt-5.4-mini",
)
print("Runtime adapter:", type(fresh.runtime).__name__)
fresh.ask("Explain prompt engineering in one sentence.")


### Same runtime choice via `ChatParams`
Useful when you want runtime selection and model settings bundled together.


In [ ]:
params = ChatParams(runtime="responses", model="gpt-5.4-mini", temperature=0.2)
teacher = Chat("You teach with short examples.", params=params)
teacher.ask("Give me a three-step checklist for my first prompt.")


## Speed taste test: continuation speed (old vs new)
This benchmark focuses on **follow-on turns in the same chat thread** (continuations).

> We warm up one turn, then measure continuation `.chat(...)` calls only.
> Note: network and model load vary. Run multiple times and compare medians.


In [ ]:
from time import perf_counter
from statistics import median

def sample_continuation_latency(runtime_selector=None, rounds=3):
    continuation_timings = []
    for _ in range(rounds):
        chat = Chat(
            "Reply in exactly seven words.",
            runtime_selector=runtime_selector,
            model="gpt-5.4-mini",
            temperature=0,
        )

        # Warm-up turn (not measured): establishes thread state.
        chat = chat.chat("Say hello to a brand-new chatsnack user.")

        # Measure follow-on requests from the same chat (continuations).
        for i in range(3):
            t0 = perf_counter()
            chat = chat.chat(f"Continuation turn {i+1}: keep it practical.")
            continuation_timings.append(perf_counter() - t0)

    return continuation_timings

classic_times = sample_continuation_latency(runtime_selector=None, rounds=3)
responses_times = sample_continuation_latency(runtime_selector="responses", rounds=3)

classic_med = median(classic_times)
responses_med = median(responses_times)
speedup = classic_med / responses_med if responses_med else float("inf")

print(f"classic continuation median:   {classic_med:.3f}s -> {classic_times}")
print(f"responses continuation median: {responses_med:.3f}s -> {responses_times}")
print(f"speedup (classic / responses): {speedup:.2f}x")


## Continue the conversation
`ask()` is a quick one-shot. `chat()` gives you a new chat with history carried forward.


In [ ]:
follow_up = teacher.chat("Now rewrite that checklist for a travel planner assistant.")
follow_up.last


## Optional: stream output
If you want token-style output, iterate `listen(...)`.


In [ ]:
for chunk in teacher.listen("Give me five tiny prompt-writing tips."):
    print(chunk, end="")
print()


---
You're ready to snack with the new Responses runtime.
